# Data-file generator — a gel block in its bath

Builds the LAMMPS data file of a finite gel block (diamond-lattice network, Kremer–Grest beads) in a solvent
bath for `simulations/slab_with_support` (the equilibration run; `slab_with_support_single.batch`).  All the
code lives in `lib/data_file_generator.py` (moved there from this notebook on 2026-10-28); this notebook only
sets the knobs.  `NETWORK` chooses the network:

* `tetrahedral` — the diamond lattice with its cube axes along the box (every gel up to 2026-10-24);
* `rot45` — the same lattice turned 45° about y, so an xz shear of the box is a cube-diagonal shear of the lattice;
* `random` — the rot45 block re-wired into a random f = 4 network by bond switching (no lattice directions; the
  isotropy check runs on the written file).

The output goes to `lammps_data_files_local/` (not in the repo).  Equilibrate it, then convert the final
configuration with `lib/add_more_plates_to_gel.py` (`--layout x_plates` for `shear_slab_two_pist`,
`six_plates` for `compress_slab`); see `docs/building_data_files.md`.


In [ ]:
import sys, importlib
from pathlib import Path
LIB = Path('lib').resolve()
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import data_file_generator as gen
gen = importlib.reload(gen)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  KNOBS -- the only cell to edit
# ══════════════════════════════════════════════════════════════════════════
NETWORK = "random"        # "tetrahedral" | "rot45" | "random"   (presets in gen.NETWORKS)
OUTPUT  = {"tetrahedral": "../../lammps_data_files_local/slab_support_5beads_tall_rho04_nowalls.data",
           "rot45":       "../../lammps_data_files_local/slab_support_5beads_tall_rho04_nowalls_rot45.data",
           "random":      "../../lammps_data_files_local/slab_support_5beads_tall_rho04_nowalls_random.data"}[NETWORK]
# the TALL ISOLATED rho04 gel (2026-10-11 / 2026-10-25): 5-bead strands, a = 1.10 x 4.8 = 5.28 sigma (the equilibrium
# lattice constant of this gel at P* = 1.5, 1.0_1.0), rho 0.40 over the whole box, 15 sigma of bath, no sheets.
# units: the aligned lattice counts CUBIC cells (9 x 9 x 21 = 47.5 x 47.5 x 110.9 sigma); the rotated builds count
# BCT cells of a/sqrt2 = 3.73 sigma along x and z (13 x 9 x 30 = 48.5 x 47.5 x 112.0 sigma), four crosslinks each.
units = {"tetrahedral": (9, 9, 21), "rot45": (13, 9, 30), "random": (13, 9, 30)}[NETWORK]
gen.build(network=NETWORK, output_file=OUTPUT,
          beads_per_chain=5, units_x=units[0], units_y=units[1], units_z=units[2],
          solvent_density=0.40,            # bath at P* = 1.5 sits at 0.48, the gel interior holds 0.22 solvent + 0.37 polymer
          side_thickness=15, below_thickness=7, top_thickness=15,   # bottom pad = below + 2 * support_thickness = 15
          support_thickness=4, support_spacing=0.2,                 # no sheets are written (include_walls=False)
          piston_margin=0, piston_exclusion=0.1, piston_height=0.8,
          pre_swell=1.10, include_walls=False)


In [ ]:
# Optional: the isotropy check on any written file (build() runs it for 'random'; the two lattices fail it, as they must)
# gen.verify_network_isotropy("../../lammps_data_files_local/slab_support_5beads_tall_rho04_nowalls_rot45.data", "rot45 lattice (reference)")
# gen.verify_network_isotropy("../../lammps_data_files_local/slab_support_5beads_tall_rho04_nowalls.data", "aligned lattice (reference)")
